# VocalCoach Colab Training

Two-stage training to solve the pitch/technique multi-task gradient conflict.

**Why this matters:** Joint training from epoch 1 causes technique gradients (~10x stronger
than pitch at epoch 1) to immediately capture the backbone, collapsing VDR to <5%.
Two-stage training with backbone freezing prevents this:

- **Stage 1** — pitch + VAD only, 50 epochs. Backbone builds clean pitch representations (target VDR ~70%).
- **Stage 2** — resume from stage 1, add technique head. Backbone is **frozen** for the first
  20 epochs so only `head_technique` trains. After epoch 20 the backbone unfreezes for joint fine-tuning.

**Checkpoint strategy:** Training writes to **local disk** (`/content/runs/`) for fast I/O.
A Drive copy cell runs once after each stage completes. If the session disconnects mid-training,
re-run setup (cells 1–4) and resume from the last Drive checkpoint.

**Run cells in order.** Cells 1–4 are setup (re-run at the start of every new session).

---
**Before starting:** upload `NanoPitch_data.zip` to `My Drive/musicalAI/vocalCoach/` on Google Drive.
```bash
cd ~/NanoPitch-MusicalAI
zip -j -1 -v NanoPitch_data.zip \
    data/clean.npz data/noise.npz data/test.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz
```

## Cell 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 64 if vram_gb > 30 else 32 if vram_gb > 15 else 16
NUM_WORKERS = 8
print(f"\nRecommended batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs('/content/data/vocalset', exist_ok=True)

# Only extract files not already present — avoids re-extracting on session resume
files_needed = {
    'clean.npz':           '/content/data/clean.npz',
    'noise.npz':           '/content/data/noise.npz',
    'test.npz':            '/content/data/test.npz',
    'technique_train.npz': '/content/data/vocalset/technique_train.npz',
    'technique_test.npz':  '/content/data/vocalset/technique_test.npz',
}
missing = [name for name, path in files_needed.items() if not os.path.exists(path)]

if missing:
    print(f"Extracting {len(missing)} file(s) from NanoPitch_data.zip...")
    with zipfile.ZipFile(f'{DRIVE_ROOT}/NanoPitch_data.zip', 'r') as z:
        for name in missing:
            dest = '/content/data/vocalset/' if 'technique' in name else '/content/data/'
            print(f"  {name}...")
            z.extract(name, dest)
else:
    print("All data files already present — skipping extraction.")

!ls -lh /content/data/
!ls -lh /content/data/vocalset/

## Cell 3 — Clone or update repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/NanoPitch-MusicalAI'
REPO_URL = 'https://github.com/YOUR_USERNAME/NanoPitch-MusicalAI'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print("Repo already cloned — fetching latest changes...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} pull origin main
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet
print("Setup complete.")

## Cell 4 — Verify data loads correctly

In [ ]:
import numpy as np

clean = np.load('/content/data/clean.npz')
print(f"clean.npz:              {list(clean.keys())}")
print(f"  mel shape:            {clean['mel'].shape}")
print(f"  clips:                {clean['lengths'].shape[0]}")

tech = np.load('/content/data/vocalset/technique_train.npz')
print(f"\ntechnique_train.npz:    {list(tech.keys())}")
print(f"  clips:                {tech['lengths'].shape[0]}")

test = np.load('/content/data/test.npz')
print(f"\ntest.npz:               {list(test.keys())}")
print(f"  clips:                {test['clips'].shape[0]}")

---
## TCN — Stage 1: pitch + VAD only (50 epochs)

Trains to **local disk** for fast checkpoint I/O. Drive copy runs after training completes.
Target by epoch 50: VDR > 60%, RPA > 96%.

**A100 optimisations vs local runs:**
- `--batch-size 64` — A100 has 80 GB VRAM, model uses <1 GB
- `--num-workers 8` — reduces CPU data-loading bottleneck
- `--seq-len 600` — longer sequences = more GPU work per step

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --output-dir /content/runs/tcn_stage1_pitchonly \
    --epochs 50 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

### TCN Stage 1 — Save to Drive
Run this after Stage 1 completes. Stage 2 reads the checkpoint from Drive so it survives session reconnects.

In [ ]:
import shutil
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
shutil.copytree(
    '/content/runs/tcn_stage1_pitchonly',
    f'{DRIVE_ROOT}/NanoPitch-runs/tcn_stage1_pitchonly',
    dirs_exist_ok=True
)
print("Stage 1 checkpoints saved to Drive.")

## TCN — Stage 2: frozen backbone (epochs 51–70) then joint fine-tuning (71–100)

Reads stage 1 checkpoint from Drive, trains to local disk, copies to Drive on completion.
Backbone frozen for 20 epochs so technique gradients cannot erode pitch representations.

Can run in a new session — re-run cells 1–4 first, checkpoint is already on Drive.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'

!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/runs/tcn_stage2_technique \
    --epochs 100 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --freeze-backbone-epochs 20 \
    --patience 0 \
    --resume {DRIVE_ROOT}/NanoPitch-runs/tcn_stage1_pitchonly/checkpoints/best_loss.pth

### TCN Stage 2 — Save to Drive

In [ ]:
import shutil
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
shutil.copytree(
    '/content/runs/tcn_stage2_technique',
    f'{DRIVE_ROOT}/NanoPitch-runs/tcn_stage2_technique',
    dirs_exist_ok=True
)
print("Stage 2 checkpoints saved to Drive.")

---
## Conformer — Stage 1: pitch + VAD only (50 epochs)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --output-dir /content/runs/conformer_stage1_pitchonly \
    --epochs 50 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

### Conformer Stage 1 — Save to Drive

In [ ]:
import shutil
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
shutil.copytree(
    '/content/runs/conformer_stage1_pitchonly',
    f'{DRIVE_ROOT}/NanoPitch-runs/conformer_stage1_pitchonly',
    dirs_exist_ok=True
)
print("Stage 1 checkpoints saved to Drive.")

## Conformer — Stage 2: frozen backbone (epochs 51–70) then joint fine-tuning (71–100)

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'

!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/runs/conformer_stage2_technique \
    --epochs 100 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --freeze-backbone-epochs 20 \
    --patience 0 \
    --resume {DRIVE_ROOT}/NanoPitch-runs/conformer_stage1_pitchonly/checkpoints/best_loss.pth

### Conformer Stage 2 — Save to Drive

In [ ]:
import shutil
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
shutil.copytree(
    '/content/runs/conformer_stage2_technique',
    f'{DRIVE_ROOT}/NanoPitch-runs/conformer_stage2_technique',
    dirs_exist_ok=True
)
print("Stage 2 checkpoints saved to Drive.")

---
## Evaluate a completed run

Reads checkpoint from Drive, evaluates locally. Change `RUN_NAME` to the run you want.

In [ ]:
RUN_NAME   = "tcn_stage2_technique"  # change to the run you want to evaluate
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'

import shutil, os
local_run = f'/content/runs/{RUN_NAME}'
os.makedirs(local_run, exist_ok=True)
shutil.copytree(
    f'{DRIVE_ROOT}/NanoPitch-runs/{RUN_NAME}',
    local_run,
    dirs_exist_ok=True
)
print("Copied. Running evaluation...")

!python vocalcoach/evaluate.py \
    --checkpoint {local_run}/checkpoints/best_metric.pth \
    --data-dir /content/data \
    --technique-dir /content/data/vocalset